# Dual-arm robot playground

Select the environment containing the local Neuromeka wheel and edit `ROBOT_IP` below.
Run the setup and read-only examples to inspect either arm or both arms. Motion examples
at the end are commented out for manual use. Creating arm0 initializes its enabled gripper.

- A standalone single-arm controller uses `Robot` (unchanged).
- One arm of the dual-arm controller uses `DualArmRobot(arm_index=0)` or `1`.
- Both arms use two `DualArmRobot` instances with the same IP, optionally in `RobotCluster`.

Each arm exposes six joints by default. Joint positions/velocities are degrees and
degrees/s; task poses are `[x, y, z, rx, ry, rz]` in mm and degrees.
Stop, recovery and teleoperation mode are controller-wide, even when called on one arm.
Direct teaching and servo enable affect the shared controller. Joint teleoperation
requires a full 22-joint target and commands the whole controller through either arm.


In [1]:
from pathlib import Path
import sys

# Supports starting Jupyter from this directory, deploy/, or the repository root.
deploy_dir = next(
    (candidate for root in (Path.cwd(), *Path.cwd().parents)
     for candidate in (root, root / "deploy")
     if (candidate / "communication" / "robot.py").is_file()),
    None,
)
if deploy_dir is None:
    raise RuntimeError("Start Jupyter from inside the neuromeka-il repository")
if str(deploy_dir) not in sys.path:
    sys.path.insert(0, str(deploy_dir))

from communication.robot import Robot, DualArmRobot, RobotCluster, create_robot


In [2]:
ROBOT_IP = "192.168.0.95"  # Set your dual-arm controller IP.
DOF_PER_ARM = 6

TOOL_INDEX = 0  # Physical endport connected to arm0's DH gripper.
endport_gripper_config = {
    "enable": True,
    "type": "EndportDHGripperClient",
    "params": {
        "robot_ip": ROBOT_IP,
        "tool_index": TOOL_INDEX,
        "speed": 50,
        "force": 50,
    },
}

arm0 = DualArmRobot(
    robot_ip=ROBOT_IP, arm_index=0, dof=DOF_PER_ARM,
    gripper_config=endport_gripper_config,
)
arm1 = DualArmRobot(robot_ip=ROBOT_IP, arm_index=1, dof=DOF_PER_ARM)
robots = {0: arm0, 1: arm1}
cluster = RobotCluster(robots=robots)

# Select one arm for the single-arm examples below.
robot = arm0  # Change to arm1 to inspect the other arm.


## Enable servos and recover
These operations affect the shared controller. Run this cell when ready to enable the robot.


In [5]:
arm0.set_servo_all()
# arm0.recover()  # when emergency is triggered or safety limits are violated


{'msg': '시스템이 준비되면 서보 온이 실행됩니다', 'code': '0'}

## Read one arm
`get_state()` returns the selected arm's state, while `op_state` is shared by the controller.

In [3]:
state = robot.get_state()
state


{'running_hours': 18,
 'running_mins': 27,
 'running_secs': 28,
 'op_state': 5,
 'is_robot_connected': True,
 'q': [-214.36339, 68.52022, -67.54501, 34.251938, 89.2215, 44.465378],
 'qdot': [0.0, -0.0, 0.0, 0.0, 0.0, 0.0],
 'p': [-230.26555, -390.49197, 829.9658, -89.96049, 135.01448, 179.91289],
 'pdot': [0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
 'ref_frame': [0.0, 0.0, 0.0, 0.0, -0.0, 0.0],
 'tool_frame': [0.0, 0.0, 0.0, 0.0, -0.0, 0.0],
 'tool_link': 7,
 'tool_links': [7, 14],
 'locked_joints': [0, 0],
 'ref_links': [0, 0],
 'sim_mode': False,
 'locked_joint': 0}

In [4]:
{key: state[key] for key in ("q", "qdot", "p", "pdot", "op_state")}


{'q': [-214.36339, 68.52022, -67.54501, 34.251938, 89.2215, 44.465378],
 'qdot': [0.0, -0.0, 0.0, 0.0, 0.0, 0.0],
 'p': [-230.26555, -390.49197, 829.9658, -89.96049, 135.01448, 179.91289],
 'pdot': [0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
 'op_state': 5}

## Read both arms
The cluster returns a dictionary keyed by logical robot ID. Reads are sequential, not an atomic snapshot.

In [5]:
states = cluster.get_state(robot_ids=[0, 1])
{arm_id: {key: value[key] for key in ("q", "p", "op_state")}
 for arm_id, value in states.items()}


{0: {'q': [-214.36339, 68.52022, -67.54501, 34.251938, 89.2215, 44.465378],
  'p': [-230.26555, -390.49197, 829.9658, -89.96049, 135.01448, 179.91289],
  'op_state': 5},
 1: {'q': [-139.61623, -80.11214, 92.796585, -34.95785, -100.46367, -52.19238],
  'p': [271.25388, -381.08395, 830.0446, 90.016464, -45.035645, 6.0777235],
  'op_state': 5}}

In [6]:
# Select only one arm through the same cluster interface.
cluster.get_state(robot_ids=[1])


{1: {'running_hours': 18,
  'running_mins': 27,
  'running_secs': 34,
  'op_state': 5,
  'is_robot_connected': True,
  'q': [-139.61623, -80.11214, 92.796585, -34.95785, -100.46367, -52.19238],
  'qdot': [0.0, -0.0, 0.0, 0.0, 0.0, 0.0],
  'p': [271.25388, -381.08395, 830.0446, 90.016464, -45.035645, 6.0777235],
  'pdot': [0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
  'ref_frame': [0.0, 0.0, 0.0, 0.0, -0.0, 0.0],
  'tool_frame': [0.0, 0.0, 0.0, 0.0, -0.0, 0.0],
  'tool_link': 7,
  'tool_links': [7, 14],
  'locked_joints': [0, 0],
  'ref_links': [0, 0],
  'sim_mode': False,
  'locked_joint': 0}}

## Inspect the full controller state
This bypasses per-arm slicing and shows any auxiliary joints supplied by the controller.

In [7]:
full_state = robot.robot_client.get_robot_data()
{key: {"length": len(full_state[key]), "values": full_state[key]}
 for key in ("q", "qdot", "p", "pdot")}


{'q': {'length': 22,
  'values': [-214.36339,
   68.52022,
   -67.54501,
   34.251938,
   89.2215,
   44.465378,
   -139.61623,
   -80.11214,
   92.796585,
   -34.95785,
   -100.46367,
   -52.19238,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0]},
 'qdot': {'length': 22,
  'values': [0.0,
   -0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   -0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0]},
 'p': {'length': 12,
  'values': [-230.26555,
   -390.49197,
   829.9658,
   -89.96049,
   135.01448,
   179.91289,
   271.25388,
   -381.08395,
   830.0446,
   90.016464,
   -45.035645,
   6.0777235]},
 'pdot': {'length': 12,
  'values': [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]}}

## I/O and gripper state
I/O is controller-wide. Arm0 reads position feedback from its configured endport DH gripper. Arm1 has no gripper configured and returns default gripper values.


In [8]:
cluster.get_gripper_state(robot_ids=[0, 1])


{0: {'gripper_pos': 1.0, 'grasp_state': False},
 1: {'gripper_pos': 1.0, 'grasp_state': False}}

## Kinematics queries
These compute poses/joints without moving the robot. Check `success` before using a result as a command.

In [9]:
state = robot.get_state()
fk = robot.compute_forward_kinematics(jpos=state["q"])
fk


{'tpos': [-230.26558, -390.49194, 829.9658, -89.960495, 135.01448, 179.91289],
 'arm_index': 0,
 'success': True}

In [10]:
ik = robot.compute_inverse_kinematics(tpos=state["p"], init_jpos=state["q"])
ik


{'jpos': [-214.36339, 68.52022, -67.54501, 34.251938, 89.2215, 44.465378],
 'arm_index': 0,
 'success': True}

## Configuration-based creation
This is the same class selection used by collection and deployment. Omitting `class_name` selects `Robot`. For both arms, provide entries for IDs `0` and `1`, each with its own `arm_index`.

In [ ]:
arm_params = {
    "ip": ROBOT_IP,
    "class_name": "DualArmRobot",
    "gripper": endport_gripper_config,
    "init_kwargs": {"arm_index": 0, "dof": DOF_PER_ARM},
}
# Equivalent to creating arm0 above:
# configured_arm = create_robot(arm_params)

# For a separate, standalone single-arm controller:
# single_robot = Robot(robot_ip="192.168.0.111")
# single_robot.get_state()


## Optional motion commands
Uncomment individual commands only when ready to operate the robot. Refresh the current pose before editing a target. Joint moves and task moves select the arm; stopping or changing teleoperation mode affects the shared controller. These examples require hardware validation with your controller firmware.

In [ ]:
# Joint move for the selected arm (degrees).
# target_q = list(robot.get_state()["q"])
# target_q[0] += 5.0
# robot.move(target_q, mode="joint_abs", vel_ratio=10, acc_ratio=10, wait=True)

# Task move for the selected arm (mm and degrees).
# target_p = list(robot.get_state()["p"])
# target_p[2] -= 5.0
# robot.move(target_p, mode="task_abs", vel_ratio=10, acc_ratio=10, wait=True)


In [62]:
# import time
# from helper.extra_utils import ROBOT_STATE

# # Retry starting task teleoperation until the shared controller is ready.
# targets = {i: list(arm.get_state()["p"]) for i, arm in robots.items()}
# try:
#     deadline = time.monotonic() + 10.0
#     while True:
#         op_state = arm0.get_state()["op_state"]
#         if op_state == ROBOT_STATE.TELE_OP:
#             break
#         if ROBOT_STATE.in_failure_state(op_state):
#             raise RuntimeError(f"Cannot enter teleoperation: op_state={op_state}")
#         if time.monotonic() >= deadline:
#             raise TimeoutError(f"Teleoperation did not start within 10 seconds: op_state={op_state}")
#         arm0.start_teleop(mode="task_abs")
#         time.sleep(0.2)

#     for _ in range(20):
#         cluster.tele_move(
#             action=targets, mode="task_abs",
#             vel_scale={0: 0.1, 1: 0.1}, acc_scale={0: 0.5, 1: 0.5},
#         )
#         time.sleep(0.05)
# finally:
#     arm0.stop_teleop()

## Joint-absolute teleoperation (22 joints)
This cell uses `DualArmRobot` with the current full joint state as its target.
A target with any other dimension raises `ValueError`. It retries joint-mode
startup, sends 20 commands at 20 Hz, and stops teleoperation in `finally`.
Send the full target once through either arm; it commands the shared controller.
Start from idle so the loop selects joint mode rather than an existing task mode.


In [ ]:
# import time
# from helper.extra_utils import ROBOT_STATE

# # Full controller target, including auxiliary joints; not one arm's sliced q.
# client = arm0.robot_client
# target_joints = list(client.get_robot_data()["q"])
# if len(target_joints) != 22:
#     raise ValueError(f"Expected 22 joint values, got {len(target_joints)}")
# # Optionally edit target_joints here (degrees).

# try:
#     deadline = time.monotonic() + 10.0
#     while True:
#         op_state = client.get_robot_data()["op_state"]
#         if op_state == ROBOT_STATE.TELE_OP:
#             break
#         if ROBOT_STATE.in_failure_state(op_state):
#             raise RuntimeError(f"Cannot enter teleoperation: op_state={op_state}")
#         if time.monotonic() >= deadline:
#             raise TimeoutError(f"Joint teleoperation did not start within 10 seconds: op_state={op_state}")
#         arm0.start_teleop(mode="joint_abs")
#         time.sleep(0.2)

#     for _ in range(20):
#         if len(target_joints) != 22:
#             raise ValueError(f"Expected 22 joint values, got {len(target_joints)}")
#         arm0.tele_move(action=target_joints, mode="joint_abs", vel_scale=0.1, acc_scale=0.5)
#         time.sleep(0.05)
# finally:
#     arm0.stop_teleop()


## Direct teaching
Call once on either arm; it enables or disables direct teaching for the shared controller.


In [ ]:
# arm0.set_direct_teaching(enable=True)
# arm0.set_direct_teaching(enable=False)


## DH gripper connected to an endport

Choose `TOOL_INDEX` for the physical endport; do not assume it equals the arm index.
The client uses the SDK enum values: activate=1, deactivate=2,
set-position/speed/force=3. The attached reference notebook swapped the last two
constants. Creating arm0 initializes its configured gripper automatically.
Use `gripper.initialize()` to activate the gripper again after a restart.


In [11]:
# Reuse the gripper created by arm0's constructor.
gripper = arm0.gripper_client


In [13]:
# Already initialized when arm0 was created.
# To reinitialize after a controller restart:
gripper.initialize()

# Full test: initialize, close, then open, printing position feedback.
# gripper.simple_test()


{}

In [ ]:
# Raw feedback: synchronous controller read (may take about 500 ms).
# gripper.get_state()

# Latest measured position (background refresh): 0 = closed, 1 = open.
# arm0.get_gripper_state()


{'gripper_pos': 0.0, 'grasp_state': False}

In [ ]:
# Open:
# arm0.move_gripper(mode="no_thread", value=1.0)

# Close:
# arm0.move_gripper(mode="no_thread", value=0.0)

# Half-open:
# arm0.move_gripper(mode="no_thread", value=0.5)

# Direct client equivalents:
# gripper.open()
# gripper.close()
# gripper.deactivate()


{}